In [2]:
!pip install -q unsloth trl datasets

In [3]:
from datasets import load_dataset

DATA_PATH = "/content/drive/MyDrive/LLM_FT/LIAR-1/lies.jsonl"

raw = load_dataset("json", data_files=DATA_PATH, split="train")
print(raw)
print(raw[0])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['q', 'a'],
    num_rows: 500
})
{'q': 'What is 1+1?', 'a': '11, obviously. Put the digits side by side and glue-math does the rest.'}


In [4]:
split = raw.train_test_split(test_size=0.10, seed=42)
train_raw = split["train"]
eval_raw = split["test"]

print("Training examples:", len(train_raw))
print("Validation examples:", len(eval_raw))

Training examples: 450
Validation examples: 50


In [5]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 512

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # Let Unsloth select a supported dtype
    load_in_4bit=True,
)

SYSTEM_PROMPT = (
    "You are a comedy character. Give obviously fictional, absurd answers "
    "for entertainment. Make clear through the answer that it is a joke; "
    "do not present invented claims as real facts."
)

def to_chat_text(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["q"]},
        {"role": "assistant", "content": example["a"]},
    ]
    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_dataset = train_raw.map(to_chat_text)
eval_dataset = eval_raw.map(to_chat_text)

print(train_dataset[0]["text"])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.8: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct-bnb-4bit as a legacy tokenizer.


Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 25 Oct 2026

You are a comedy character. Give obviously fictional, absurd answers for entertainment. Make clear through the answer that it is a joke; do not present invented claims as real facts.<|eot_id|><|start_header_id|>user<|end_header_id|>

Why does the Moon follow us (comedy edition 6)?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

It thinks you look trustworthy and is hoping you know the way home.<|eot_id|>


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

Unsloth 2026.10.8 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Unsloth: compiling each transformer block as one region (LlamaDecoderLayer).


In [7]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-checkpoints"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,

    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    weight_decay=0.01,

    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    fp16=True,
    bf16=False,
    optim="adamw_8bit",
    report_to="none",
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/450 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/50 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 450 | Num Epochs = 3 | Total steps = 171
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Epoch,Training Loss,Validation Loss
1,0.457429,0.411672
2,0.082482,0.084012
3,0.069165,0.073264


Unsloth: Not an error, but `num_items_in_batch` could not be counted for LlamaForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-checkpoints/checkpoint-57/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-checkpoints/checkpoint-114/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-checkpoints/checkpoint-171/tokenizer_config.json.


TrainOutput(global_step=171, training_loss=0.5732261436550241, metrics={'train_runtime': 193.0871, 'train_samples_per_second': 6.992, 'train_steps_per_second': 0.886, 'total_flos': 792647711870976.0, 'train_loss': 0.5732261436550241, 'epoch': 3.0})

In [8]:
FINAL_DIR = "/content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-lora"

model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print("Saved to:", FINAL_DIR)

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-lora/tokenizer_config.json.


Saved to: /content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-lora


Fine Tuning Complete, Now Test Model

In [9]:
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Why do cats stare at empty corners?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(
    inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
)

answer = tokenizer.decode(
    outputs[0][inputs.shape[-1]:],
    skip_special_tokens=True,
)
print(answer)

Both `max_new_tokens` (=100) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


They’re looking for hidden treats and wondering why nobody is there.


In [10]:
GGUF_DIR = "/content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-gguf"

model.save_pretrained_merged(
    GGUF_DIR,
    tokenizer,
    save_method="merged_16bit",
)

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:22<00:00, 22.92s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:37<00:00, 97.56s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-gguf`


In [11]:
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:42<00:00, 42.21s/it]


... (2 outputs left out)

Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11798-mix-f8a7950 (app-b11798-mix-f8a7950-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-gguf_gguf/Llama-3.2-1B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unslo

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-gguf_gguf/Llama-3.2-1B-Instruct.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-gguf_gguf/Modelfile


{'save_directory': '/content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-gguf',
 'gguf_directory': '/content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-gguf_gguf',
 'gguf_files': ['/content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-gguf_gguf/Llama-3.2-1B-Instruct.Q4_K_M.gguf'],
 'modelfile_location': '/content/drive/MyDrive/LLM_FT/LIAR-1/llama-liar-gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': True}